In [ ]:
import pandas as pd 
import numpy as np

### Prepare questions

In [ ]:
## Load all the questions and labels

df = pd.read_csv("Datasets\\MedChangeQA.csv")

questions = df["Question"].astype(str).str.strip().tolist()
labels_n = df["Newest Label"].astype(str).str.strip().tolist()
labels_o = df["Outdated Label"].astype(str).str.strip().tolist()

In [ ]:
import time 

## Prepare the messages for answering the questions by each LLM.
## Most LLMs were called through Together AI API or hosted locally and queried.

all_messages = list()
for idx in range(len(questions)):

    system_message = "You're a helpful AI assistant helping answer clinical and medical questions based on your best knowledge."
    user_message_begin = f'''
           Please answer this clinical question only with SUPPORTED (if the question is supported by the clinical research) or REFUTED (if the hypothesis is refuted by the current clinical research) or NOT ENOUGH INFORMATION (if there is insufficient evidence for the question in current research). Please give your output in form of LABEL: (label) . Briefly explain your answer.

    QUESTION:
    '''

    finalstring = ""
    finalstring += user_message_begin
    finalstring += questions[idx]  + "\n"

    messages =  [  
            {'role':'system', 
             'content': system_message},    
            {'role':'user', 
             'content': f"{finalstring}"},  
    ] 
    
    all_messages.append(messages)

### Ollama setup for retrieving model

In [ ]:
import os, urllib.request
import subprocess
subprocess.run(["pip", "install", "-q", "zstandard"], check=True)
import zstandard as zstd
import tarfile

# Download the official Ollama archive 
archive_url = "https://ollama.com/download/ollama-linux-amd64.tar.zst"
temp_archive = os.path.expanduser("~/ollama-temp.tar.zst")
target_dir = os.path.expanduser("~/.local")

urllib.request.urlretrieve(archive_url, temp_archive)
print("Downloading Ollama package...")

# Decompress zstandard and extract tar directly to ~/.local
print("Extracting package to ~/.local ...")
dctx = zstd.ZstdDecompressor()
with open(temp_archive, "rb") as ifh:
    with dctx.stream_reader(ifh) as reader:
        with tarfile.open(fileobj=reader, mode="r|") as tar:
            tar.extractall(path=target_dir)

# Cleanup temporary archive and grant executable permissions
if os.path.exists(temp_archive):
    os.remove(temp_archive)

ollama_bin = os.path.expanduser("~/.local/bin/ollama")
os.chmod(ollama_bin, 0o755)

# Add to PATH and verify
if os.path.expanduser("~/.local/bin") not in os.environ["PATH"]:
    os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

print("Installed successfully:")
print(subprocess.getoutput("ollama --version"))

In [ ]:
# start Ollama background daemon
user_bin = os.path.expanduser("~/.local/bin")
if user_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{user_bin}:{os.environ['PATH']}"

print("Starting background Ollama service...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# wait 
for _ in range(10):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://localhost:11434") as res:
            print("Status:", res.read().decode().strip())
            break
    except Exception:
        pass

In [ ]:
! ollama pull qwen2.5:7b

### Collect responses

In [ ]:
from ollama import chat

def query_llm(prompt):
    response = chat(
        model='qwen2.5:7b',
        messages=prompt,
        options={
            'temperature': 0,
            'num_predict': 512,  
        }
    )
    return response


start_time = time.time()
with open("qwen25-7b_answers.txt", "w") as f:
    curr_idx = 0

    for msgs in all_messages:
        response = query_llm(msgs)
        result = response.message.content.strip()
        result = result.replace("\n", " ||| ")
        
        f.write(result + "\n")
        curr_idx += 1

        # Track time.
        if curr_idx % 1000 == 0:
            end_time = time.time()
            print(curr_idx, end_time-start_time)
            start_time = time.time()